# Softmax 函數的「贏者通吃」特性示範
##前言：使用Claude的Prompt
>"我們要讓學生有一個看 softmax 有那種「贏者通吃」, 也就是因為指數放大, 所以最高分很容易會過度放大。請寫一個要給 Colab 的示範程式, 因為要給學生練習的, 請一段一段寫, 並且用 Markdown 在文字儲存格中放入好的說明。請要寫一個做 softmax 的函式, 然後用 ipython 的 interact 讓學生觀察, 自己輸入, 比如三個數字, 經 softmax 後的結果。"請一段一段的寫出來
Md語法與py語法不要混在一起。

## 第一段：匯入必要的函式庫

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, fixed
import ipywidgets as widgets
from IPython.display import display

# 更穩定的中文字體設置方法
import subprocess
import sys
import os
import matplotlib.font_manager as fm

def setup_chinese_fonts():
    """設置中文字體"""
    try:
        print("正在安裝中文字體...")
        subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-noto-cjk'], check=True, capture_output=True)
        try:
            if hasattr(fm, '_rebuild'):
                fm._rebuild()
            else:
                fm.fontManager.__init__()
        except:
            pass
        available_fonts = [f.name for f in fm.fontManager.ttflist]
        chinese_fonts = [f for f in available_fonts if 'Noto' in f and ('CJK' in f or 'Sans' in f)]
        if chinese_fonts:
            plt.rcParams['font.family'] = [chinese_fonts[0]]
            print(f"成功設置字體: {chinese_fonts[0]}")
            return True
        else:
            raise Exception("未找到合適的中文字體")
    except Exception as e:
        print(f"字體安裝失敗: {e}")
        print("使用英文標題以避免顯示問題")
        plt.rcParams['font.family'] = ['DejaVu Sans']
        return False

chinese_font_available = setup_chinese_fonts()
plt.rcParams['axes.unicode_minus'] = False
print("函式庫匯入完成！")

---

## 第二段：什麼是 Softmax？

Softmax 函數是機器學習中常用的激活函數，特別用在多分類問題的輸出層。它的數學定義是：

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$$

**Softmax 的重要特性：**
1. 輸出值都在 0 到 1 之間
2. 所有輸出值的總和等於 1（可視為機率分佈）
3. 具有「贏者通吃」特性：最大的輸入值會被過度放大

In [ ]:
print("Softmax 概念示範：")
print("假設有三個分數：[1, 2, 3]")
print("直接正規化：[1/6, 2/6, 3/6] =", [1/6, 2/6, 3/6])
print("經過 exp() 後：[e¹, e², e³] ≈", [np.exp(1), np.exp(2), np.exp(3)])
print("可以看到差距被放大了！")


---

## 第三段：實作 Softmax 函數

In [ ]:
def softmax(x):
    """計算 softmax 函數"""
    x = np.array(x, dtype=float)
    x_stable = x - np.max(x)
    exp_x = np.exp(x_stable)
    softmax_probs = exp_x / np.sum(exp_x)
    return softmax_probs

test_input = [1, 2, 3]
print(f"輸入: {test_input}")
print(f"Softmax 輸出: {softmax(test_input)}")
print(f"總和: {np.sum(softmax(test_input)):.6f}")

---

## 第四段：視覺化函數

In [ ]:
def visualize_softmax(values, title="Softmax 比較"):
    softmax_values = softmax(values)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    if chinese_font_available:
        title1, title2 = '原始輸入值', 'Softmax 輸出（機率）'
        xlabel, ylabel1, ylabel2 = '項目', '數值', '機率'
    else:
        title1, title2 = 'Original Input Values', 'Softmax Output (Probability)'
        xlabel, ylabel1, ylabel2 = 'Item', 'Value', 'Probability'
    x_pos = range(len(values))
    ax1.bar(x_pos, values, color='skyblue', alpha=0.7)
    ax1.set_title(title1)
    ax1.set_xlabel(xlabel)
    ax1.set_ylabel(ylabel1)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([f'x{i+1}' for i in range(len(values))])
    for i, v in enumerate(values):
        y_offset = max(values)*0.02 if max(values) > 0 else 0.1
        ax1.text(i, v + y_offset, f'{v:.2f}', ha='center', va='bottom')
    ax2.bar(x_pos, softmax_values, color='lightcoral', alpha=0.7)
    ax2.set_title(title2)
    ax2.set_xlabel(xlabel)
    ax2.set_ylabel(ylabel2)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f'x{i+1}' for i in range(len(values))])
    ax2.set_ylim(0, 1)
    for i, v in enumerate(softmax_values):
        ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
    plt.tight_layout()
    plt.show()
    print("=" * 50)
    print(f"原始值: {values}")
    print(f"Softmax: {[f'{x:.3f}' for x in softmax_values]}")
    print(f"最大值位置: x{np.argmax(values) + 1}")
    print(f"最大機率: {np.max(softmax_values):.3f}")
    print("=" * 50)

test_values = [2, 3, 7]
visualize_softmax(test_values)

---

## 第五段：互動式示範 - 三個數字的情況

In [ ]:
def interactive_softmax_3_numbers(x1, x2, x3):
    values = [x1, x2, x3]
    visualize_softmax(values)

print("🎯 請調整下面的滑桿，觀察 Softmax 如何處理不同的輸入值")
print("注意觀察當某個數值比其他數值大很多時會發生什麼！")
print("-" * 50)
from ipywidgets import interactive, VBox
import IPython.display as display
widget = interactive(interactive_softmax_3_numbers,
                    x1=FloatSlider(value=1.0, min=-5.0, max=10.0, step=0.1, description='x1:'),
                    x2=FloatSlider(value=2.0, min=-5.0, max=10.0, step=0.1, description='x2:'),
                    x3=FloatSlider(value=3.0, min=-5.0, max=10.0, step=0.1, description='x3:'))
widget.children[-1].clear_output()
display.display(widget)

---

## 第六段：展示「贏者通吃」特性

In [ ]:
def demonstrate_winner_takes_all():
    print("🔍 「贏者通吃」特性示範")
    print("=" * 50)
    print("\n情況 1: 數值相近")
    values1 = [2.0, 2.1, 2.2]
    softmax1 = softmax(values1)
    print(f"輸入: {values1}")
    print(f"Softmax: {[f'{x:.3f}' for x in softmax1]}")
    print(f"最大機率佔比: {np.max(softmax1):.1%}")
    print("\n情況 2: 有一個明顯較大的值")
    values2 = [2.0, 2.1, 5.0]
    softmax2 = softmax(values2)
    print(f"輸入: {values2}")
    print(f"Softmax: {[f'{x:.3f}' for x in softmax2]}")
    print(f"最大機率佔比: {np.max(softmax2):.1%}")
    print("\n情況 3: 差距更大")
    values3 = [2.0, 2.1, 8.0]
    softmax3 = softmax(values3)
    print(f"輸入: {values3}")
    print(f"Softmax: {[f'{x:.3f}' for x in softmax3]}")
    print(f"最大機率佔比: {np.max(softmax3):.1%}")
    print("\n" + "=" * 50)
    print("💡 觀察：隨著最大值與其他值的差距增加，")
    print("   最大值對應的機率會急劇增加，接近 100%！")
    print("   這就是 softmax 的「贏者通吃」特性。")
demonstrate_winner_takes_all()

---

## 第七段：溫度參數的影響
🌡️ 溫度參數示範

溫度高 → 分佈更平均

溫度低 → 分佈更集中（更強的贏者通吃）

In [ ]:
def softmax_with_temperature(x, temperature=1.0):
    x = np.array(x, dtype=float)
    x_scaled = x / temperature
    x_stable = x_scaled - np.max(x_scaled)
    exp_x = np.exp(x_stable)
    return exp_x / np.sum(exp_x)

def interactive_temperature_demo(x1, x2, x3, temperature):
    values = [x1, x2, x3]
    softmax_values = softmax_with_temperature(values, temperature)
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    if chinese_font_available:
        title = f'溫度參數 = {temperature:.1f} 的 Softmax'
        xlabel, ylabel = '項目', '機率'
    else:
        title = f'Softmax with Temperature = {temperature:.1f}'
        xlabel, ylabel = 'Item', 'Probability'
    x_pos = range(len(values))
    ax.bar(x_pos, softmax_values, color='orange', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'x{i+1}' for i in range(len(values))])
    ax.set_ylim(0, 1)
    for i, v in enumerate(softmax_values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
    plt.tight_layout()
    plt.show()
    print(f"原始值: {values}")
    print(f"溫度 T = {temperature:.1f}")
    print(f"Softmax: {[f'{x:.3f}' for x in softmax_values]}")

print("🌡️ 溫度參數示範")
temp_label = '溫度:' if chinese_font_available else 'Temperature:'
from ipywidgets import interactive, VBox
import IPython.display as display
temp_widget = interactive(interactive_temperature_demo,
                         x1=FloatSlider(value=2.0, min=-3.0, max=5.0, step=0.1, description='x1:'),
                         x2=FloatSlider(value=3.0, min=-3.0, max=5.0, step=0.1, description='x2:'),
                         x3=FloatSlider(value=4.0, min=-3.0, max=5.0, step=0.1, description='x3:'),
                         temperature=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description=temp_label))
temp_widget.children[-1].clear_output()
display.display(temp_widget)

---

## 總結

Softmax 函數的「贏者通吃」特性來自於：
1. **指數函數的性質**：$e^x$ 會放大較大的數值
2. **相對差距的放大**：即使原始值差距不大，經過指數運算後差距會被顯著放大
3. **正規化過程**：讓最大的項目獲得絕大部分的機率質量

這個特性在深度學習中很重要，特別是在分類問題中，我們希望模型對最可能的類別給出高信心度的預測。

---
##心得
透過這次的測試，得知`softmax`的一些特性：第一，最大值會被**過度放大**，尤其是**原始值的差距較大時**，使得轉換結果後的差距更大；第二，若原始值**有負有正**，則正值會被明顯的放大；第三，若原始值都是負的或都是正的，結果**相差不多**；第四，`temperature`越小，差距更大